# Assignment 4: Transformer Tuning with Ray Tune & Optuna
**Roll No: B22CH045**  
English-to-Hindi translation hyperparameter tuning.

In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import math
import os
import pickle
from collections import Counter
from torch.utils.data import Dataset, DataLoader

In [2]:
# Load data (same as baseline)
df = pd.read_csv('/kaggle/input/datasets/realyogesh/english-hindi-tsv/English-Hindi.tsv', sep='\t', header=None, names=["id1", "en", "id2", "hi"])
df = df[["en", "hi"]]
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"Total pairs: {len(df)}")

Total pairs: 13186


In [3]:
# Load vocabs from baseline run (or build if not present)
if os.path.exists("en_vocab.pkl") and os.path.exists("hi_vocab.pkl"):
    with open("en_vocab.pkl", "rb") as f:
        en_vocab = pickle.load(f)
    with open("hi_vocab.pkl", "rb") as f:
        hi_vocab = pickle.load(f)
    print(f"Loaded vocabs: EN={len(en_vocab)}, HI={len(hi_vocab)}")
else:
    from collections import Counter
    class Vocabulary:
        def __init__(self, freq_threshold=2):
            self.freq_threshold = freq_threshold
            self.itos = {0: "<pad>", 1: "<sos>", 2: "<eos>", 3: "<unk>"}
            self.stoi = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}
            self.idx = 4
        def build_vocab(self, sentence_list):
            frequencies = Counter()
            for sentence in sentence_list:
                for word in sentence.lower().strip().split():
                    frequencies[word] += 1
            for word, freq in frequencies.items():
                if freq >= self.freq_threshold:
                    self.stoi[word] = self.idx
                    self.itos[self.idx] = word
                    self.idx += 1
        def numericalize(self, sentence):
            tokens = sentence.lower().strip().split()
            return [self.stoi.get(t, self.stoi["<unk>"]) for t in tokens]
        def __len__(self): return len(self.stoi)
        def __getitem__(self, token): return self.stoi.get(token, self.stoi["<unk>"])
    en_vocab = Vocabulary(freq_threshold=2)
    hi_vocab = Vocabulary(freq_threshold=2)
    en_vocab.build_vocab(df["en"].tolist())
    hi_vocab.build_vocab(df["hi"].tolist())
    print(f"Built vocabs: EN={len(en_vocab)}, HI={len(hi_vocab)}")

Built vocabs: EN=4117, HI=4044


In [4]:
MAX_LEN = 50
D_MODEL = 512  # fixed so that num_heads 4 or 8 divides it

def encode_sentence(sentence, vocab, max_len=50):
    tokens = [vocab.stoi["<sos>"]] + vocab.numericalize(sentence)[:max_len-2] + [vocab.stoi["<eos>"]]
    return tokens + [vocab.stoi["<pad>"]] * (max_len - len(tokens))

class TranslationDataset(Dataset):
    def __init__(self, df, en_vocab, hi_vocab, max_len=50):
        self.en_sentences = df["en"].tolist()
        self.hi_sentences = df["hi"].tolist()
        self.en_vocab, self.hi_vocab = en_vocab, hi_vocab
        self.max_len = max_len
    def __len__(self): return len(self.en_sentences)
    def __getitem__(self, idx):
        src = encode_sentence(self.en_sentences[idx], self.en_vocab, self.max_len)
        tgt = encode_sentence(self.hi_sentences[idx], self.hi_vocab, self.max_len)
        return torch.tensor(src), torch.tensor(tgt)

def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_batch = torch.stack(src_batch)
    tgt_batch = torch.stack(tgt_batch)
    tgt_input = tgt_batch[:, :-1]
    tgt_output = tgt_batch[:, 1:]
    return src_batch, tgt_input, tgt_output

In [5]:
# --- Transformer model classes (same as baseline) ---
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model, self.num_heads = d_model, num_heads
        self.d_k = d_model // num_heads
        self.query_linear = nn.Linear(d_model, d_model)
        self.key_linear = nn.Linear(d_model, d_model)
        self.value_linear = nn.Linear(d_model, d_model)
        self.out_linear = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, q, k, v, mask=None):
        B = q.size(0)
        Q = self.query_linear(q).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.key_linear(k).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.value_linear(v).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        if mask is not None: scores = scores.masked_fill(mask == 0, -1e9)
        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(self.dropout(attn), V).transpose(1, 2).contiguous().view(B, -1, self.d_model)
        return self.out_linear(out)

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=2048, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
    def forward(self, x): return self.linear2(self.dropout(self.relu(self.linear1(x))))

class LayerNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))
        self.eps = eps
    def forward(self, x):
        mean, std = x.mean(-1, keepdim=True), x.std(-1, keepdim=True)
        return self.gamma * (x - mean) / (std + self.eps) + self.beta

class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1, self.norm2 = LayerNorm(d_model), LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, mask=None):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, mask)))
        return self.norm2(x + self.dropout(self.ffn(x)))

class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1, self.norm2, self.norm3 = LayerNorm(d_model), LayerNorm(d_model), LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, enc_out, src_mask=None, tgt_mask=None):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, tgt_mask)))
        x = self.norm2(x + self.dropout(self.cross_attn(x, enc_out, enc_out, src_mask)))
        return self.norm3(x + self.dropout(self.ffn(x)))

class Encoder(nn.Module):
    def __init__(self, input_vocab_size, d_model, num_layers, num_heads, d_ff, max_len, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(input_vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, mask=None):
        x = self.dropout(self.pos_enc(self.embed(x)))
        for layer in self.layers: x = layer(x, mask)
        return x

class Decoder(nn.Module):
    def __init__(self, target_vocab_size, d_model, num_layers, num_heads, d_ff, max_len, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(target_vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, enc_out, src_mask=None, tgt_mask=None):
        x = self.dropout(self.pos_enc(self.embed(x)))
        for layer in self.layers: x = layer(x, enc_out, src_mask, tgt_mask)
        return x

class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, num_layers=6, num_heads=8, d_ff=2048, max_len=100, dropout=0.1):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, d_model, num_layers, num_heads, d_ff, max_len, dropout)
        self.decoder = Decoder(tgt_vocab_size, d_model, num_layers, num_heads, d_ff, max_len, dropout)
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)
    def make_pad_mask(self, seq, pad_idx): return (seq != pad_idx).unsqueeze(1).unsqueeze(2)
    def make_subsequent_mask(self, size): return torch.tril(torch.ones((size, size))).bool().to(next(self.parameters()).device)
    def forward(self, src, tgt, src_pad_idx, tgt_pad_idx):
        src_mask = self.make_pad_mask(src, src_pad_idx)
        tgt_pad_mask = self.make_pad_mask(tgt, tgt_pad_idx)
        tgt_sub_mask = self.make_subsequent_mask(tgt.size(1))
        tgt_mask = tgt_pad_mask & tgt_sub_mask
        enc_out = self.encoder(src, src_mask)
        dec_out = self.decoder(tgt, enc_out, src_mask, tgt_mask)
        return self.fc_out(dec_out)

In [6]:
!pip install ray[tune] optuna

import optuna  # import first (important)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 2.8 MB/s eta 0:00:00


In [7]:
from ray import tune
# import ray.train as train
from ray.tune.search.optuna import OptunaSearch
from ray.tune.schedulers import ASHAScheduler
from ray.air import session
from ray.tune import Checkpoint 
import tempfile
import torch.cuda.amp                  

In [8]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SRC_PAD_IDX = en_vocab["<pad>"]
TGT_PAD_IDX = hi_vocab["<pad>"]
SRC_VOCAB_SIZE = len(en_vocab)
TGT_VOCAB_SIZE = len(hi_vocab)

In [9]:
def train_tune(config):
    num_epochs = config.get("num_epochs", 30)
    lr = config["lr"]
    batch_size = config["batch_size"]
    num_heads = config["num_heads"]
    d_ff = config["d_ff"]
    dropout = config["dropout"]
    d_model = D_MODEL

    dataset = TranslationDataset(df, en_vocab, hi_vocab, max_len=MAX_LEN)
    train_loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=2,          
        pin_memory=True,        
        persistent_workers=True 
    )

    model = Transformer(
        src_vocab_size=SRC_VOCAB_SIZE,
        tgt_vocab_size=TGT_VOCAB_SIZE,
        d_model=d_model,
        num_layers=6,
        num_heads=num_heads,
        d_ff=d_ff,
        max_len=MAX_LEN,
        dropout=dropout
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss(ignore_index=TGT_PAD_IDX)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # ✅ Mixed precision setup
    # scaler = torch.cuda.amp.GradScaler()

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0

        for src, tgt_input, tgt_output in train_loader:
            src = src.to(DEVICE, non_blocking=True)        # ✅ non_blocking with pin_memory
            tgt_input = tgt_input.to(DEVICE, non_blocking=True)
            tgt_output = tgt_output.to(DEVICE, non_blocking=True)

            # ✅ Mixed precision forward pasa
            with torch.amp.autocast('cuda', dtype=torch.bfloat16, enabled=True):
                output = model(src, tgt_input, SRC_PAD_IDX, TGT_PAD_IDX)
                output = output.view(-1, output.shape[-1])
                tgt_flat = tgt_output.view(-1)
                loss = criterion(output, tgt_flat)

            optimizer.zero_grad()
            loss.backward()   # ✅ scaled backward
            optimizer.step()

            epoch_loss += loss.item()

        mean_loss = epoch_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {mean_loss:.4f}")

        # ✅ Checkpoint inside loop so ASHA can prune
        with tempfile.TemporaryDirectory() as tmpdir:
            path = os.path.join(tmpdir, "checkpoint.pt")
            torch.save({
                "model_state": model.state_dict(),
                "epoch": epoch + 1,
                "loss": mean_loss,
            }, path)
            checkpoint = Checkpoint.from_directory(tmpdir)
            tune.report({"loss": mean_loss, "epoch": epoch + 1}, checkpoint=checkpoint)

In [10]:
# Search space: at least 4 hyperparameters (lr, batch_size, num_heads, d_ff, dropout)
# D_MODEL=512 so num_heads must divide it: 4 or 8
param_space = {
    "lr": tune.loguniform(1e-5, 1e-3),
    "batch_size": tune.choice([32, 64]),
    "num_heads": tune.choice([4, 8]),
    "d_ff": tune.choice([1024, 2048]),
    "dropout": tune.uniform(0.1, 0.4),
    "num_epochs": 30  # Efficiency: cap at 30 epochs per trial (vs baseline 100)
}

In [11]:
def test_train_tune_report_fix(config):
    """A simplified function to test the tune.report API call."""
    # Simulate some dummy metrics and epoch
    dummy_loss = 0.5
    dummy_epoch = 1

    # Simulate checkpoint creation
    with tempfile.TemporaryDirectory() as tmpdir:
        path = os.path.join(tmpdir, "checkpoint.pt")
        # Create a dummy file to ensure Checkpoint.from_directory works
        with open(path, "w") as f:
            f.write("dummy content")
        checkpoint = Checkpoint.from_directory(tmpdir)

        # This is the line we want to test
        tune.report({"loss": dummy_loss, "epoch": dummy_epoch}, checkpoint=checkpoint)

    print(f"Successfully reported metrics: loss={dummy_loss}, epoch={dummy_epoch}")

# Define a minimal search space for the test
test_param_space = {
    "lr": tune.loguniform(1e-5, 1e-3),
    "batch_size": tune.choice([16]),
}

# Configure a minimal tuner to run the test function once
test_tuner = tune.Tuner(
    test_train_tune_report_fix,
    tune_config=tune.TuneConfig(
        num_samples=1,
    ),
    param_space=test_param_space,
)

print("Running quick test for tune.report fix...")
try:
    test_results = test_tuner.fit()
    print("Quick test completed successfully! The tune.report syntax is correct.")
except Exception as e:
    print(f"Quick test failed with an error: {e}")

2026-03-18 16:20:14,894	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/root/ray_results/test_train_tune_report_fix_2026-03-18_16-19-14' in 0.0044s.
2026-03-18 16:20:14,899	INFO tune.py:1041 -- Total run time: 50.25 seconds (4.78 seconds for the tuning loop).


Quick test completed successfully! The tune.report syntax is correct.


(test_train_tune_report_fix pid=428) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/root/ray_results/test_train_tune_report_fix_2026-03-18_16-19-14/test_train_tune_report_fix_3a9f8_00000_0_batch_size=16,lr=0.0000_2026-03-18_16-20-10/checkpoint_000000)


(test_train_tune_report_fix pid=428) Successfully reported metrics: loss=0.5, epoch=1


In [12]:
optuna_search = OptunaSearch(metric="loss", mode="min")
asha = ASHAScheduler(max_t=30, grace_period=5, reduction_factor=2, time_attr="epoch", metric="loss", mode="min")

from ray import tune

trainable = tune.with_resources(
    train_tune,
    resources={"cpu": 2, "gpu": 1}
)

tuner = tune.Tuner(
    trainable,   # 👈 use wrapped trainable
    tune_config=tune.TuneConfig(
        search_alg=optuna_search,
        num_samples=20,
        scheduler=asha,
        # metric="loss",              # ✅ single source of truth
        # mode="min",
    ),
    param_space=param_space,
)

results = tuner.fit()

(train_tune pid=473) [2026-03-18 16:21:26,744 E 473 498] core_worker_process.cc:842: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(train_tune pid=514) [2026-03-18 16:21:33,302 E 514 543] core_worker_process.cc:842: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14


(train_tune pid=514) Epoch 1/30, Loss: 5.8213


(train_tune pid=514) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/root/ray_results/train_tune_2026-03-18_16-20-53/train_tune_a54cd1a0_2_batch_size=32,d_ff=1024,dropout=0.1583,lr=0.0000,num_epochs=30,num_heads=4_2026-03-18_16-20-59/checkpoint_000000)


(train_tune pid=473) Epoch 1/30, Loss: 5.8083


(train_tune pid=473) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/root/ray_results/train_tune_2026-03-18_16-20-53/train_tune_1505cc0c_1_batch_size=32,d_ff=2048,dropout=0.1897,lr=0.0000,num_epochs=30,num_heads=4_2026-03-18_16-20-53/checkpoint_000000)


...

(train_tune pid=3263) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/root/ray_results/train_tune_2026-03-18_16-20-53/train_tune_776e8f3b_20_batch_size=64,d_ff=1024,dropout=0.2705,lr=0.0002,num_epochs=30,num_heads=8_2026-03-18_19-29-56/checkpoint_000027)


(train_tune pid=3263) Epoch 29/30, Loss: 0.5071


(train_tune pid=3263) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/root/ray_results/train_tune_2026-03-18_16-20-53/train_tune_776e8f3b_20_batch_size=64,d_ff=1024,dropout=0.2705,lr=0.0002,num_epochs=30,num_heads=8_2026-03-18_19-29-56/checkpoint_000028)


(train_tune pid=3263) Epoch 30/30, Loss: 0.4729


(train_tune pid=3263) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/root/ray_results/train_tune_2026-03-18_16-20-53/train_tune_776e8f3b_20_batch_size=64,d_ff=1024,dropout=0.2705,lr=0.0002,num_epochs=30,num_heads=8_2026-03-18_19-29-56/checkpoint_000029)
2026-03-18 20:11:50,332	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/root/ray_results/train_tune_2026-03-18_16-20-53' in 0.0202s.
2026-03-18 20:11:50,342	INFO tune.py:1041 -- Total run time: 13857.11 seconds (13857.02 seconds for the tuning loop).


In [13]:
best_result = results.get_best_result(metric="loss", mode="min")
print("Best config:", best_result.config)
print("Best loss:", best_result.metrics.get("loss"))
print("Best epoch:", best_result.metrics.get("epoch"))

Best config: {'lr': 0.00024121778613652668, 'batch_size': 64, 'num_heads': 8, 'd_ff': 1024, 'dropout': 0.27048235481128247, 'num_epochs': 30}
Best loss: 0.47288614425106323
Best epoch: 30


In [14]:
# Load best checkpoint and save as required filename (B22CH045_ass_4_best_model.pth)
import glob
checkpoint_dirs = sorted(glob.glob(os.path.join(best_result.path, "checkpoint_*")), key=lambda p: int(p.split("_")[-1]) if p.split("_")[-1].isdigit() else 0)
state, best_epoch, best_loss = None, 0, 0.0
if checkpoint_dirs:
    latest_dir = checkpoint_dirs[-1]
    ckpt_file = os.path.join(latest_dir, "checkpoint.pt")
    if os.path.exists(ckpt_file):
        ckpt = torch.load(ckpt_file, map_location=DEVICE)
        best_epoch = ckpt.get("epoch", 0)
        best_loss = ckpt.get("loss", 0)
        state = ckpt["model_state"]

if state is not None:
    cfg = best_result.config
    model = Transformer(
        src_vocab_size=SRC_VOCAB_SIZE,
        tgt_vocab_size=TGT_VOCAB_SIZE,
        d_model=D_MODEL,
        num_layers=6,
        num_heads=cfg["num_heads"],
        d_ff=cfg["d_ff"],
        max_len=MAX_LEN,
        dropout=cfg["dropout"]
    ).to(DEVICE)
    model.load_state_dict(state)
    torch.save(model.state_dict(), "B22CH045_ass_4_best_model.pth")
    print(f"Saved B22CH045_ass_4_best_model.pth (epoch={best_epoch}, loss={best_loss:.4f})")
else:
    print("No checkpoint found; run the tuning cell and ensure checkpoints are written.")

Saved B22CH045_ass_4_best_model.pth (epoch=30, loss=0.4729)


In [15]:
# Optional: Evaluate BLEU on best model (same val set as baseline for report)
def translate_sentence(model, sentence, en_vocab, hi_vocab, max_len=50):
    model.eval()
    tokens = encode_sentence(sentence, en_vocab, max_len=max_len)
    src_tensor = torch.tensor(tokens).unsqueeze(0).to(DEVICE)
    tgt_tokens = [hi_vocab["<sos>"]]
    for _ in range(max_len):
        tgt_tensor = torch.tensor(tgt_tokens).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            output = model(src_tensor, tgt_tensor, SRC_PAD_IDX, TGT_PAD_IDX)
        next_token = output[0, -1].argmax().item()
        tgt_tokens.append(next_token)
        if next_token == hi_vocab["<eos>"]:
            break
    translated = [hi_vocab.itos[idx] for idx in tgt_tokens[1:-1]]
    return " ".join(translated)

val_dataset = [
    ("I love you.", "मैं तुमसे प्यार करता हूँ।"),
    ("How are you?", "आप कैसे हैं?"),
    ("You should sleep.", "आपको सोना चाहिए।"),
    ("Maybe Tom doesn't love you.", "टॉम शायद तुमसे प्यार नहीं करता है।"),
    ("Let me tell Tom.", "मुझे टॉम को बताने दीजिए।"),
]
import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
smoothie = SmoothingFunction().method4
references = [[ref.split()] for _, ref in val_dataset]
hypotheses = [translate_sentence(model, en, en_vocab, hi_vocab).split() for en, _ in val_dataset]
bleu_score = corpus_bleu(references, hypotheses, smoothing_function=smoothie)
print(f"Best model BLEU (same val set as baseline): {bleu_score * 100:.2f}")

Best model BLEU (same val set as baseline): 66.25
